# Sensitivity analysis for Lag (different HRF)
- Takes into account interaction of groups

In [1]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az

import matplotlib.pyplot as plt

# set random seed for reproducibility
random_seed = 42

In [2]:
def fit_lag_interaction(coupling_vals, label):
    with pm.Model() as m:
        beta_coupling = pm.Normal('beta_coupling', 0, 1)
        beta_amg      = pm.Normal('beta_amg', 0, 1)
        beta_trialNo  = pm.Normal('beta_trialNo', 0, 1)
        beta_group_raw = pm.Normal('beta_group_raw', 0, 1, shape=n_groups-1)
        beta_group = pm.math.concatenate([[0], beta_group_raw])
        beta_interaction_raw = pm.Normal('beta_interaction_raw', 0, 1, shape=n_groups-1)
        beta_interaction = pm.math.concatenate([[0], beta_interaction_raw])
        mu_a = pm.Normal('mu_a', 0, 1); sigma_a = pm.HalfNormal('sigma_a', 1)
        z_a = pm.Normal('z_a', 0, 1, shape=n_subs)
        a = pm.Deterministic('a', mu_a + z_a*sigma_a)
        mu = (a[sub_idx] + beta_group[group_idx] + beta_coupling*coupling_vals
              + beta_interaction[group_idx]*coupling_vals + beta_amg*amg + beta_trialNo*trialNo)
        sigma = pm.HalfNormal('sigma', 1)
        pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
        tr = pm.sample(chains=4, random_seed=42, return_inferencedata=True,
                       idata_kwargs={"log_likelihood": True})
    post = tr.posterior
    sHC  = post['beta_coupling']
    sVCC = post['beta_coupling'] + post['beta_interaction_raw'][:,:,0]
    sPTS = post['beta_coupling'] + post['beta_interaction_raw'][:,:,1]
    diff = sVCC - sPTS
    print(f"\n[{label}]")
    for nm, s in [('HC',sHC),('VCC',sVCC),('VPTSD',sPTS)]:
        v=s.values.ravel(); print(f"  {nm}: {v.mean():+.3f} HDI={az.hdi(v,hdi_prob=0.89)}")
    v=diff.values.ravel()
    print(f"  VCC-VPTSD: {v.mean():+.3f} HDI={az.hdi(v,hdi_prob=0.89)} pd={ (v<0).mean()*100:.1f}%")
    return tr

In [4]:
# read lag 0
df = pd.read_csv('data/coupling_hrf0.csv')
df.head()

,sub,trialNo,condition,coupling,amg,amg_vmpfc,source,hrf,pe,group
0,sub-010,1,CSplusUS1,0.857143,0.037732,0.690476,A,0,0.500000,VCC
1,sub-010,2,CSminus1,0.714286,0.197202,0.571429,A,0,-0.500000,VCC
2,sub-010,3,CSplus1,0.642857,0.095099,0.333333,A,0,-0.575084,VCC
3,sub-010,4,CSplusUS1,0.309524,0.141916,0.333333,A,0,0.516641,VCC
4,sub-010,5,CSminus1,0.857143,0.045487,0.523810,A,0,-0.424916,VCC


In [5]:
df['sub_idx'] = pd.Categorical(df['sub']).codes
n_subs = df['sub_idx'].nunique()

# Encode 'group' as integer indices (make ordering explicit!)
# Data uses: HC (healthy controls), VCC (combat controls), VPTSD (PTSD)
group_order = ['HC', 'VCC', 'VPTSD']
df['group'] = pd.Categorical(df['group'], categories=group_order, ordered=True)
df['group_idx'] = df['group'].cat.codes
n_groups = df['group_idx'].nunique()

# Check which group is reference (index 0)
print("Group coding (0 = reference):", {g: i for i, g in enumerate(df['group'].cat.categories)})

# Extract variables
pe = df['pe'].values
coupling = df['coupling'].values
amg = df['amg'].values
trialNo = df['trialNo'].values
sub_idx = df['sub_idx'].values
group_idx = df['group_idx'].values

Group coding (0 = reference): {'HC': 0, 'VCC': 1, 'VPTSD': 2}


In [6]:
tr_amg = fit_lag_interaction(coupling, label='amygdala')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, beta_group_raw, beta_interaction_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 12 seconds.



[amygdala]
  HC: +0.094 HDI=[0.03604042 0.14343367]
  VCC: +0.087 HDI=[0.03567145 0.13485797]
  VPTSD: +0.127 HDI=[0.07527195 0.1763996 ]
  VCC-VPTSD: -0.040 HDI=[-0.1086253   0.03656538] pd=81.4%


In [7]:
tr_amg_vmpfc = fit_lag_interaction(df['amg_vmpfc'], label='amygdala')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, beta_group_raw, beta_interaction_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 9 seconds.



[amygdala]
  HC: -0.050 HDI=[-0.09683304 -0.00372741]
  VCC: -0.099 HDI=[-0.14179775 -0.06164463]
  VPTSD: +0.018 HDI=[-0.0210665   0.06303539]
  VCC-VPTSD: -0.118 HDI=[-0.17713723 -0.06041164] pd=99.9%


# Now lag 4 seconds (tr=2)

In [8]:
df = pd.read_csv('data/coupling_hrf2.csv')
df.head()

,sub,trialNo,condition,coupling,amg,amg_vmpfc,source,hrf,pe,group
0,sub-010,1,CSplusUS1,0.809524,-0.027128,0.785714,A,2,0.500000,VCC
1,sub-010,2,CSminus1,0.666667,0.183908,0.523810,A,2,-0.500000,VCC
2,sub-010,3,CSplus1,0.809524,0.216987,0.190476,A,2,-0.575084,VCC
3,sub-010,4,CSplusUS1,0.285714,-0.047552,0.642857,A,2,0.516641,VCC
4,sub-010,5,CSminus1,0.523810,0.038555,-0.214286,A,2,-0.424916,VCC


In [9]:
df['sub_idx'] = pd.Categorical(df['sub']).codes
n_subs = df['sub_idx'].nunique()

# Encode 'group' as integer indices (make ordering explicit!)
# Data uses: HC (healthy controls), VCC (combat controls), VPTSD (PTSD)
group_order = ['HC', 'VCC', 'VPTSD']
df['group'] = pd.Categorical(df['group'], categories=group_order, ordered=True)
df['group_idx'] = df['group'].cat.codes
n_groups = df['group_idx'].nunique()

# Check which group is reference (index 0)
print("Group coding (0 = reference):", {g: i for i, g in enumerate(df['group'].cat.categories)})

# Extract variables
pe = df['pe'].values
coupling = df['coupling'].values
amg = df['amg'].values
trialNo = df['trialNo'].values
sub_idx = df['sub_idx'].values
group_idx = df['group_idx'].values

Group coding (0 = reference): {'HC': 0, 'VCC': 1, 'VPTSD': 2}


In [10]:
tr_amg_4 = fit_lag_interaction(coupling, label='amygdala')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, beta_group_raw, beta_interaction_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 11 seconds.



[amygdala]
  HC: +0.079 HDI=[0.02597943 0.1338663 ]
  VCC: +0.064 HDI=[0.01831303 0.11972732]
  VPTSD: +0.078 HDI=[0.03148924 0.12583241]
  VCC-VPTSD: -0.014 HDI=[-0.08407121  0.05431854] pd=62.8%


In [11]:
tr_amg_vmpfc_4 = fit_lag_interaction(df['amg_vmpfc'], label='amygdala')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, beta_group_raw, beta_interaction_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 9 seconds.



[amygdala]
  HC: -0.012 HDI=[-0.05379612  0.03500716]
  VCC: -0.062 HDI=[-0.10051177 -0.02403868]
  VPTSD: +0.025 HDI=[-0.01420541  0.06488258]
  VCC-VPTSD: -0.086 HDI=[-0.14094558 -0.02975994] pd=99.4%
